განზომილებები დეშბორდის ფილტრებისა და კავშირებისთვის.

ვარსკვლავური სქემა: ფაქტების ცხრილი CalcMembers (სტრიქონი = ერთი მთამსვლელი ერთ ექსპედიციაში) და სამი განზომილება, რომლებიც მას გასაღებით უკავშირდება.

CalcDimPeak  -> PeakId -> CalcMembers.PeakId
CalcDimYear  -> ExpeditionYear -> CalcMembers.ExpeditionYear
CalcDimNation -> Nation -> CalcMembers.Citizenship

კავშირები დეშბორდში ეწერება Data Model -> Relationships-ში.
სააგენტოების განზომილება აქ აღარაა: მას CalcAgencyPerformance ანაცვლებს 03-ში, რომელსაც მოცულობის გარდა წარმატებისა და სიკვდილიანობის მაჩვენებლებიც აქვს.


In [0]:
%sql
-- 1. CalcDimPeak — მწვერვალები (16)

CREATE OR REPLACE TABLE getdata.calculated.CalcDimPeak AS
SELECT
    Pk.PeakId,
    Pk.PeakName,
    Pk.HeightMetres,
    Pk.HeightBand,
    Pk.MountainRange,
    Pk.Region,
    Pk.ClimbingStatus,
    Pk.FirstAscentYear,
    Pk.FirstAscentCountry
FROM getdata.calculated.CalcPeaks AS Pk;

COMMENT ON TABLE getdata.calculated.CalcDimPeak IS
    'მწვერვალების განზომილება დეშბორდისთვის. გასაღები: PeakId.';

In [0]:
%sql
-- 2. CalcDimYear — წლები, ათწლეულები, ეპოქები
-- მხოლოდ ის წლები, როცა ექსპედიცია ნამდვილად შედგა.

CREATE OR REPLACE TABLE getdata.calculated.CalcDimYear AS
SELECT DISTINCT
    Ex.ExpeditionYear,
    Ex.Decade,
    CONCAT(CAST(Ex.Decade AS STRING), '-იანები') AS DecadeLabel,
    Ex.Era
FROM getdata.calculated.CalcExpeditions AS Ex;

COMMENT ON TABLE getdata.calculated.CalcDimYear IS
    'წლების განზომილება ათწლეულითა და ეპოქით. გასაღები: ExpeditionYear.';

In [0]:
%sql
-- 3. CalcDimNation — მოქალაქეობები

CREATE OR REPLACE TABLE getdata.calculated.CalcDimNation AS
WITH NationVolume AS (
    SELECT
        Mb.Citizenship,
        COUNT(*) AS ClimberCount
    FROM getdata.calculated.CalcMembers AS Mb
    WHERE Mb.Citizenship IS NOT NULL
    GROUP BY Mb.Citizenship
)
SELECT
    Nt.Citizenship AS Nation,
    Nt.ClimberCount,
    CASE
        WHEN Nt.ClimberCount >= 1000 THEN 'დიდი (1000+)'
        WHEN Nt.ClimberCount >= 100 THEN 'საშუალო (100–999)'
        ELSE 'მცირე (<100)'
    END AS NationSizeBand,
    Nt.Citizenship = 'Georgia' AS IsGeorgia
FROM NationVolume AS Nt;

COMMENT ON TABLE getdata.calculated.CalcDimNation IS
    'მოქალაქეობების განზომილება. გასაღები: Nation (= CalcMembers.Citizenship). IsGeorgia გამოყოფს ქართველ მთამსვლელებს.';


In [0]:
%sql
-- კავშირების შემოწმება:
-- ყოველ ფაქტს უნდა ჰქონდეს შესაბამისი განზომილება.
-- სამივე რიცხვი 0 უნდა იყოს.

SELECT
    SUM(CASE WHEN Dp.PeakId IS NULL THEN 1 ELSE 0 END) AS MembersWithoutPeak,
    SUM(CASE WHEN Dy.ExpeditionYear IS NULL THEN 1 ELSE 0 END) AS MembersWithoutYear,
    SUM(
        CASE
            WHEN Mb.Citizenship IS NOT NULL
                 AND Dn.Nation IS NULL
            THEN 1
            ELSE 0
        END
    ) AS MembersWithoutNation
FROM getdata.calculated.CalcMembers AS Mb
LEFT JOIN getdata.calculated.CalcDimPeak AS Dp
    ON Mb.PeakId = Dp.PeakId
LEFT JOIN getdata.calculated.CalcDimYear AS Dy
    ON Mb.ExpeditionYear = Dy.ExpeditionYear
LEFT JOIN getdata.calculated.CalcDimNation AS Dn
    ON Mb.Citizenship = Dn.Nation;